# 08 — DistilBERT Inference Demo

In this notebook, we load the final saved DistilBERT model
and use it to predict sentiment for new, unseen movie reviews.

Pipeline:

Raw text
→ Tokenizer
→ DistilBERT
→ Logits
→ Probabilities
→ Sentiment

In [2]:
from pathlib import Path

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

In [3]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print(
    "Project root:",
    PROJECT_ROOT,
)

Project root: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers


In [5]:
MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "distilbert_imdb"
)

assert MODEL_DIR.exists()

print(
    "Model directory:",
    MODEL_DIR,
)

Model directory: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers\models\distilbert_imdb


In [6]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device,
)

Device: cuda


In [7]:
tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_DIR
    )
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_DIR
    )
    .to(device)
)

model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [9]:
MAX_LENGTH = 128

In [10]:
@torch.no_grad()
def predict_sentiment(
    text: str,
) -> dict:

    model.eval()

    encoded = tokenizer(
        text,
        padding=False,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(device)
        for key, value
        in encoded.items()
    }

    outputs = model(
        **encoded
    )

    logits = outputs.logits

    probabilities = torch.softmax(
        logits,
        dim=1,
    )

    predicted_class_id = (
        probabilities
        .argmax(dim=1)
        .item()
    )

    negative_probability = (
        probabilities[
            0, 0
        ]
        .item()
    )

    positive_probability = (
        probabilities[
            0, 1
        ]
        .item()
    )

    predicted_label = (
        model.config.id2label[
            predicted_class_id
        ]
    )

    return {
        "label": predicted_label,
        "class_id": predicted_class_id,
        "negative_probability": (
            negative_probability
        ),
        "positive_probability": (
            positive_probability
        ),
    }

In [11]:
example_review = (
    "This movie was absolutely fantastic. "
    "I loved every minute of it."
)

result = predict_sentiment(
    example_review
)

result

{'label': 'pos',
 'class_id': 1,
 'negative_probability': 0.017052125185728073,
 'positive_probability': 0.9829478859901428}

In [12]:
print(
    "Review:"
)

print(
    example_review
)

print()

print(
    "Predicted sentiment:",
    result["label"],
)

print(
    "Negative probability:",
    f"{result['negative_probability']:.4f}",
)

print(
    "Positive probability:",
    f"{result['positive_probability']:.4f}",
)

Review:
This movie was absolutely fantastic. I loved every minute of it.

Predicted sentiment: pos
Negative probability: 0.0171
Positive probability: 0.9829


In [13]:
negative_review = (
    "The movie was boring, predictable, "
    "and painfully slow. I hated it."
)

predict_sentiment(
    negative_review
)

{'label': 'neg',
 'class_id': 0,
 'negative_probability': 0.9919053316116333,
 'positive_probability': 0.008094606921076775}

In [14]:
tricky_review = (
    "I thought this movie would be terrible, "
    "but it was actually surprisingly good."
)

predict_sentiment(
    tricky_review
)

{'label': 'pos',
 'class_id': 1,
 'negative_probability': 0.03924486041069031,
 'positive_probability': 0.9607551693916321}

In [15]:
another_tricky_review = (
    "The movie started really well, "
    "but the ending was so bad that "
    "I cannot recommend it."
)

predict_sentiment(
    another_tricky_review
)

{'label': 'neg',
 'class_id': 0,
 'negative_probability': 0.9913187623023987,
 'positive_probability': 0.008681187406182289}

In [16]:
def show_prediction(
    text: str,
) -> None:

    result = predict_sentiment(
        text
    )

    print(
        "Text:"
    )

    print(
        text
    )

    print()

    print(
        "Prediction:",
        result["label"],
    )

    print(
        "Negative:",
        f"{result['negative_probability']:.2%}",
    )

    print(
        "Positive:",
        f"{result['positive_probability']:.2%}",
    )

In [17]:
show_prediction(
    "An amazing film with great acting "
    "and a wonderful story."
)

Text:
An amazing film with great acting and a wonderful story.

Prediction: pos
Negative: 1.36%
Positive: 98.64%


In [21]:
import pandas as pd


In [22]:
@torch.no_grad()
def predict_sentiment_batch(
    texts: list[str],
) -> pd.DataFrame:

    model.eval()

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(device)
        for key, value
        in encoded.items()
    }

    outputs = model(
        **encoded
    )

    probabilities = torch.softmax(
        outputs.logits,
        dim=1,
    )

    predicted_class_ids = (
        probabilities
        .argmax(dim=1)
        .cpu()
        .numpy()
    )

    probabilities = (
        probabilities
        .cpu()
        .numpy()
    )

    predicted_labels = [
        model.config.id2label[
            int(class_id)
        ]
        for class_id
        in predicted_class_ids
    ]

    return pd.DataFrame(
        {
            "text": texts,
            "prediction": predicted_labels,
            "negative_probability": (
                probabilities[:, 0]
            ),
            "positive_probability": (
                probabilities[:, 1]
            ),
        }
    )

In [23]:
demo_reviews = [
    "Absolutely brilliant movie. I loved it.",
    "Terrible film. A complete waste of time.",
    "It started badly but became really enjoyable.",
    "The acting was good, but the story was disappointing.",
]

demo_predictions = (
    predict_sentiment_batch(
        demo_reviews
    )
)

demo_predictions

,text,prediction,negative_probability,positive_probability
0,Absolutely brilliant movie. I loved it.,pos,0.019580,0.980420
1,Terrible film. A complete waste of time.,neg,0.995272,0.004728
2,It started badly but became really enjoyable.,pos,0.085665,0.914335
3,"The acting was good, but the story was disappo...",neg,0.978904,0.021096


In [24]:
demo_predictions[
    "confidence"
] = (
    demo_predictions[
        [
            "negative_probability",
            "positive_probability",
        ]
    ]
    .max(axis=1)
)

demo_predictions

,text,prediction,negative_probability,positive_probability,confidence
0,Absolutely brilliant movie. I loved it.,pos,0.019580,0.980420,0.980420
1,Terrible film. A complete waste of time.,neg,0.995272,0.004728,0.995272
2,It started badly but became really enjoyable.,pos,0.085665,0.914335,0.914335
3,"The acting was good, but the story was disappo...",neg,0.978904,0.021096,0.978904


In [25]:
ambiguous_review = (
    "The movie was okay. "
    "Some parts were good and "
    "some parts were boring."
)

show_prediction(
    ambiguous_review
)

Text:
The movie was okay. Some parts were good and some parts were boring.

Prediction: neg
Negative: 91.41%
Positive: 8.59%


In [26]:
test_result = predict_sentiment(
    "I really enjoyed this movie."
)

assert (
    test_result["label"]
    in {"neg", "pos"}
)

assert (
    0.0
    <= test_result[
        "negative_probability"
    ]
    <= 1.0
)

assert (
    0.0
    <= test_result[
        "positive_probability"
    ]
    <= 1.0
)

probability_sum = (
    test_result[
        "negative_probability"
    ]
    +
    test_result[
        "positive_probability"
    ]
)

assert abs(
    probability_sum - 1.0
) < 1e-5

print(
    "Inference pipeline checks passed."
)

Inference pipeline checks passed.


## Final inference pipeline

The final DistilBERT inference pipeline is:

New raw review
→ Saved tokenizer
→ Token IDs and attention mask
→ Saved fine-tuned DistilBERT
→ Two logits
→ Softmax
→ Negative and positive probabilities
→ Final sentiment prediction

No training or fitting happens during inference.